# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI, by de-duplicating from Andersen, and relabeling sequences. 

In [1]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [60]:
# Paths

# Dates
start_date = "06-14-2025"
end_date = "07-04-2025"
date_range = start_date + "--" + end_date

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = home + "NCBI_Virus/downloads/" + date_range + "_Antarctica_North_America_South_America/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_Antarctica_North_America_South_America/" 

andersen = home + "Andersen/complete/11-01-2021--06-13-2025/" # Use the previous date range for de-duplication!
combined_files = home + "Combinations/Andersen_NCBI_Virus/" + date_range + "_Antarctica_North_America_South_America/" 

references = "C:/Users/maksi/Documents/Statistics/projects/Avian_Flu/references/"
# references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

genotypes_df = pd.read_excel("genotype_key.xlsx")
genotypes = list(genotypes_df["Genotype"])

# genotypes = ["B3.13", "D1.1"]

os.chdir(downloads)

## De-Duplication

In [61]:
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.SRA_Accession, as_index=False).size()
# print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="SRA_Accession")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="first") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
# metadata_segments

2622


In [62]:
# De-duplicate from Andersen using SRA Accession

# If even one SRA Accession in this list exists in the NCBI Virus dataframe, remove it from NCBI Virus dataframe
andersen_sras = []
# Grab files
for dirpath, dirs, files in os.walk(andersen):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
            sra_accessions = fasta_file["Identifier"]
            for value in sra_accessions.values:
                if "SRR" in value:
                    andersen_sras.append(value)
            
# Remove duplicates from Andersen
for value in andersen_sras: # to remove
    metadata_segments = metadata_segments[metadata_segments["SRA_Accession"] != value]
    # print(value)
    
print(len(metadata_segments))
# metadata_segments
# print(count)

40


## Add sequences to dataframe

In [63]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title

# Get sequences and headers together
headers = []
isolates = []
sras = []
headers_seqs = {}

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

# Filter dataframe to only include filtered SRA accessions
sequences_fasta = sequences_fasta[sequences_fasta["full_header"].str.contains("|".join(list(metadata_segments["SRA_Accession"].values)))]
sequences_fasta["SRA_Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-4])
# Extract segment number so that we can add the correct sequences to the correct sample
sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-1]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on=["SRA_Accession", "Segment"])

40
40


In [64]:
# metadata_segments

## Create FASTA files using deduplicated sequences

In [ ]:
# Create 1 fasta file per header
metadata_segments["Partial_Header"] = metadata_segments["full_header"].apply(lambda x: "|".join(x.split("|")[:-1]))

# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments[metadata_segments["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        df_list.append(df)

# Make fasta files
for df in df_list:
    # Forbidden characters in file name 
    title = df["Partial_Header"].values[0]
    for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
        title = title.replace(c, "_")
    df_to_fasta(df, title + ".fasta", temp_files)

## Re-Labeling Using GenoFlu

In [23]:
# Merging

os.chdir(downloads)

output_genoflu = pd.read_csv("output.tsv", delimiter="\t")

# print(output_genoflu)

# print(metadata_seqs["Header"])

file_name = metadata_seqs["Header"].apply(lambda x: x.split("|")[-1].replace(" ", "_") + "_temp.fasta")
for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
    file_name = file_name.apply(lambda x: x.replace(c, ""))

metadata_seqs["File Name"] = file_name

metadata_genoflu = metadata_seqs.merge(output_genoflu, how="left", on="File Name")

metadata_genoflu = metadata_genoflu.ffill()



print(metadata_genoflu)


       Accession GenBank_RefSeq SRA_Accession     BioSample    BioProject  \
0     PV845802.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
1     PV845803.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
2     PV845804.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
3     PV845805.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
4     PV845806.1        GenBank   SRR33681813  SAMN48710861  PRJNA1102327   
...          ...            ...           ...           ...           ...   
2219  PV792621.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2220  PV792622.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2221  PV792623.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2222  PV792624.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   
2223  PV792625.1        GenBank   SRR33476664  SAMN48398150   PRJNA980729   

          Organism_Name                         Species Genotype_x  \
0    

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [32]:
# Re-Labeling

os.chdir(home)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

fix_animals_andersen(metadata_genoflu, animals_ref)

metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# print(metadata_genoflu["Host_Type"])

names = ">" + metadata_genoflu["SRA_Accession"] + "|A/" + metadata_genoflu["Host"] + "/" + metadata_genoflu["Geo_Location"].apply(lambda x: x.split(": ")[-1]) + "/" + metadata_genoflu["Isolate"] + "/" + metadata_genoflu["Years"].apply(lambda x: str(x)) + "|" + metadata_genoflu["Genotype_x"] + "|" + metadata_genoflu["Geo_Location"].apply(lambda x: x.replace(": ", "-")) + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype_y"]

metadata_genoflu["Name"] = names

print(metadata_genoflu["Name"])


0       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
1       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
2       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
3       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
4       >SRR33681813|A/bos taurus/CA/24-028985-001-til...
                              ...                        
2219    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
2220    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
2221    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
2222    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
2223    >SRR33476664|A/gallus gallus/NY/25-013275-010-...
Name: Name, Length: 2224, dtype: object


## Merge all datasets and de-duplicate again

In [33]:
# # Set up segments

# segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}

# metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].map(segments)

# # print(metadata_genoflu["Segment_Name"])

# # Separate into several dataframes based on genotype + segment
# segment_genotype_dfs = []
# for segment in segments.values():
#     print(segment)
#     m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
#     print(m_g)
#     for genotype in list(set(m_g["Genotype_y"].values)):
#         if "Not assigned:" not in genotype:
#             df = m_g[(m_g["Genotype_y"] == genotype)] # & (metadata_genoflu["Segment"] == segment)]
#             segment_genotype_dfs.append(df)
#             # pair = genotype + "_" + segment
#             # print(pair)

# print(segment_genotype_dfs[3])

# # print(metadata_genoflu["Header"].apply(lambda x: x.split("|")[-1].split("/")[-1].split(")")[2:]))

In [34]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}



metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].map(segments)

# print(metadata_genoflu["Segment_Name"])

# Separate into several dataframes based on genotype + segment
# b313_count = 0
segment_genotype_dfs = []
for segment in segments.values():
    # print(segment)
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    # print(m_g)
    for genotype in list(set(m_g["Genotype_y"].values)):
        # if "B3.13" in genotype:
        if genotype in genotypes:
            # b313_count += 1
            df = m_g[(m_g["Genotype_y"] == genotype)] # & (metadata_genoflu["Segment"] == segment)]
            # b313_count += len(df)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

# print(b313_count)
# print(segment_genotype_dfs[8])

B3.6_PB2
B3.13_PB2
D1.1_PB2
B3.6_PB1
B3.13_PB1
D1.1_PB1
B3.6_PA
B3.13_PA
D1.1_PA
B3.6_HA
B3.13_HA
D1.1_HA
B3.6_NP
B3.13_NP
D1.1_NP
B3.6_NA
B3.13_NA
D1.1_NA
B3.6_MP
B3.13_MP
D1.1_MP
B3.6_NS
B3.13_NS
D1.1_NS


In [35]:
# Create FASTA files

os.chdir(complete_files)

names = []

for df in segment_genotype_dfs:
    if len(df["Genotype_y"].values[0]) > 0:
        file_name = df["Genotype_y"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            names.append(name)
            # print(name)
            sequence = df.loc[index, "Sequence"]
            # print(sequence)
            # break 
        # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        output_file.close()

print(len(names))

2216


In [36]:
print(segment_genotype_dfs)

[       Accession GenBank_RefSeq SRA_Accession     BioSample   BioProject  \
1656  PV847514.1        GenBank   SRR33682399  SAMN48711133  PRJNA980729   

          Organism_Name                         Species Genotype_x  \
1656  Influenza A virus  Alphainfluenzavirus influenzae       H5N1   

                     Isolate  Segment  ... Genotype_y  \
1656  25-014787-002-original        1  ...       B3.6   

                            Genotype List Used, >=98.0%  \
1656  PB2:am5, PB1:am4, PA:ea1, HA:ea1, NP:am1.4.1, ...   

                             Genotype Sample Title List  \
1656  am5:23-001855-001:PB2, am4:23-001855-001:PB1, ...   

                            Genotype Percent Match List  \
1656  99.08%, 98.86%, 98.42%, 98.30%, 98.93%, 98.37%...   

              Genotype Mismatch List Genotype Average Depth of Coverage List  \
1656  21, 26, 34, 29, 16, 23, 15, 11       Ran on FASTA - No Coverage Report   

     Host_Type Years                                               Name 

In [37]:
# Concatenate

os.chdir(combined_files)

filenames_gisaid_andersen = []
for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
    for dirpath, dirs, files in os.walk(gisaid_andersen): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            filenames_gisaid_andersen.append(file_name)
        break 

print(filenames_gisaid_andersen)

filenames_ncbi = []
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi.append(file_name)
    break 

print(filenames_ncbi)

do_not_do_these_partials = []
for ga_file in filenames_gisaid_andersen:
    partial_filename_ga = ga_file.split("_")[-4].split("/")[-1] + "_" + ga_file.split("_")[-3]
    for nv_file in filenames_ncbi:
        partial_filename_nv = nv_file.split("_")[-3].split("/")[-1] + "_" + nv_file.split("_")[-2]
        if partial_filename_ga == partial_filename_nv:
            do_not_do_these_partials.append(partial_filename_ga)
            print(partial_filename_nv)
            filenames = [ga_file, nv_file]
            with open(combined_files + partial_filename_ga + "_combined_" + date_range + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)

for ga_file in filenames_gisaid_andersen:
    partial_filename_ga = ga_file.split("_")[-4].split("/")[-1] + "_" + ga_file.split("_")[-3]
    if partial_filename_ga not in do_not_do_these_partials:
        with open(combined_files + partial_filename_ga + "_combined_" + date_range + ".fasta", 'w') as outfile2:
            # for fname in filenames_gisaid_andersen:
            with open(ga_file) as infile1:
                for line in infile1:
                    outfile2.write(line)
    

# lines = 0

# fns_seen = set()
# for filename in filenames_ncbi:
#     print("NCBI_Virus", filename)
#     # lines = 0
#     # filename.split("_")[-5] + "." +   
#     partial_filename = filename.split("_")[-3].split("/")[-1] + "_" + filename.split("_")[-2] 
#     print(partial_filename)
    
#     for fn in filenames_gisaid_andersen:
#         # print(fns_seen)
#         if partial_filename in fn: # and partial_filename not in fns_seen: # If partial filenames match
#             fns_seen.add(fn)
#         #     fns_seen.add(partial_filename)
#             # print("hi", partial_filename)
#             filenames = [filename, fn]
#             with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'w') as outfile:
#                 for fname in filenames:
#                     with open(fname) as infile1:
#                         for line in infile1:
#                             outfile.write(line)

# for fn in filenames_gisaid_andersen: # Extra files
#     if fn not in fns_seen:
#         partial_filename = fn.split("_")[-4].split("/")[-1] + "_" + fn.split("_")[-3] 
#         with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'w') as outfile:
#             for fname in filenames:
#                 with open(fname) as infile1:
#                     for line in infile1:
#                         outfile.write(line)
    
#                             # if line[0] == ">":
#                             #     lines += 1
#             #         infile.close()
#             # outfile.close()
#         # elif partial_filename not in fns_seen: # Save the ones that aren't shared -- if partial filename was not seen before
#         #     # print("hello", partial_filename)
#         #     fns_seen.add(partial_filename)
#         #     with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'w') as outfile2:
#         #         # Copy
#         #         with open(fn) as infile2:
#         #             for line in infile2:
#         #                 outfile2.write(line)
#         # else: # If partial filename doesn't match AND if partial filename has been seen before
#         #     continue 
#         #                 # if line[0] == ">":
#         #                 #     lines += 1
#         #             # infile.close()
#         #         with open(fn) as infile3:
#         #             for line in infile3:
#         #                 outfile2.write(line)
#         #                 # if line[0] == ">":
#                         #     lines += 1
#             #     infile.close()
#         outfile.close()


#     # break 

#     # print(lines)

# # lines = 0
# # Counting
#     with open(combined_files + partial_filename + "_combined_" + update_date + ".fasta", 'r') as outfile:
#         for line in outfile.readlines():
#             if line[0] == ">":
#                 lines += 1

# print(lines)

['C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_HA_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_MP_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_NA_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_NP_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genotypes/06-14-2025--06-27-2025_all_genotypes/A3_NS_combined_06-14-2025--06-27-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/GISAID_Andersen/all_genoty

In [38]:
# # Mega fasta file
# all_files = []
# for dirpath, dirs, files in os.walk(combined_files):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         all_files.append(file_name)

# with open(combined_files + "all_files_combined_" + update_date + ".fasta", 'w') as outfile3:
#     for fname in all_files:
#         with open(fname) as infile4:
#             for line in infile4:
#                 outfile3.write(line)
# #         infile.close()
# # outfile.close()

In [40]:
# Animals 

os.chdir(home)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['gallus gallus', 'anatidae', 'bos taurus', 'perdicinae', 'meleagris gallopavo']
[]
                      avian               cattle        feline   other_mammal  \
0          great_horned_owl            dairy_cow           cat     deer mouse   
1              common_raven               cattle  domestic_cat    house_mouse   
2             cooper's_hawk  cattle milk product     feral_cat          skunk   
3              coopers_hawk          bovine_milk        feline  striped_skunk   
4                   peafowl              bovine   domestic-cat     norway rat   
..                      ...                  ...           ...            ...   
798  von_schrenck's_bittern                  NaN           NaN            NaN   
799           harris's_hawk                  NaN           NaN            NaN   
800           eurasian_coot                  NaN           NaN            NaN   
801                   layer                  NaN           NaN            NaN   
802              perdicin

In [ ]:
# # Concat

# os.chdir(combined_files)

# for dirpath, dirs, files in os.walk(complete_files):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         print(file_name)
#         for dirpath1, dirs1, files1 in os.walk(gisaid_andersen):
#             for file1 in files1:
#                 file_name1 = os.path.join(dirpath1, file1)
#                 if file_name.split("/")[-1].split("_")[0] + file_name.split("/")[-1].split("_")[1] == file_name1.split("/")[-1].split("_")[0] + file_name1.split("/")[-1].split("_")[1]:
                    
#                     output_path = combined_files + file_name1.split("/")[-1] # Genotype and Segment should all be the same
#                     output_file = open(output_path, "w")
#                     with open(file_name) as f:
#                         for line in f.readlines():
#                             output_file.write(line)
#                             # output_file.write("\n")
#                         f.close()
#                     with open(file_name1) as f1:
#                        for line in f1.readlines():
#                             output_file.write(line)
#                             # output_file.write("\n")
#                     f1.close()  

#                     output_file.close()